In [15]:
# 导入必要的库
import matplotlib as mpl
import matplotlib.pyplot as plt
# 在Jupyter notebook中内联显示图表
%matplotlib inline  
import numpy as np
import sklearn
import pandas as pd
import os
import sys
import time
from tqdm.auto import tqdm  # 进度条库
import torch
import torch.nn as nn
import torch.nn.functional as F

# 打印Python版本信息
print(sys.version_info)

# 打印各个库的版本信息
for module in mpl, np, pd, sklearn, torch:
    print(module.__name__, module.__version__)
    
# 设置设备：如果有GPU则使用GPU，否则使用CPU
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print(device)


sys.version_info(major=3, minor=12, micro=12, releaselevel='final', serial=0)
matplotlib 3.10.8
numpy 2.3.5
pandas 3.0.0
sklearn 1.8.0
torch 2.10.0+cu130
cuda:0


# 数据预处理

In [16]:
# INSERT_YOUR_CODE
from torchvision import datasets, transforms
import os

# 设置数据目录
data_dir = './archive/'

# 定义预处理: resize到128x128, 转为tensor
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4363, 0.4328, 0.3291], std=[0.2427, 0.2382, 0.2413])
])

# 读取训练集和测试集
train_dataset = datasets.ImageFolder(root=os.path.join(data_dir, 'training'), transform=transform)
test_dataset = datasets.ImageFolder(root=os.path.join(data_dir, 'validation'), transform=transform)

# 获取类别名（方便后续显示标签）
class_names = train_dataset.classes
class_names

['n0', 'n1', 'n2', 'n3', 'n4', 'n5', 'n6', 'n7', 'n8', 'n9']

In [17]:
len(train_dataset)

1097

In [18]:
train_dataset[0][0].shape #特征

torch.Size([3, 128, 128])

In [19]:
train_dataset[0][1] #标签

0

In [20]:
len(test_dataset)

272

# 计算test_dataset的均值和标准差

In [21]:
# from torch.utils.data import DataLoader
# import torch

# loader = DataLoader(train_dataset, batch_size=128, shuffle=False, num_workers=2)

# mean = torch.zeros(3)
# std = torch.zeros(3)
# n_pixels = 0

# for images, _ in loader:  # images: [B, 3, 128, 128]
#     batch_pixels = images.numel() // 3  # total pixels per channel
#     mean += images.sum(dim=[0, 2, 3])
#     std  += (images ** 2).sum(dim=[0, 2, 3])
#     n_pixels += batch_pixels

# mean /= n_pixels
# std = torch.sqrt(std / n_pixels - mean ** 2)

# print("按通道均值:", mean)
# print("按通道标准差:", std)

In [22]:
from torch.utils.data import DataLoader

# 创建训练集和验证集的DataLoader
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,  # 训练时打乱数据
    num_workers=2  # 使用多进程加载数据
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,  # 测试时不需要打乱
    num_workers=2
)

print(f"训练集DataLoader批次数: {len(train_loader)}")
print(f"测试集DataLoader批次数: {len(test_loader)}")
print(f"每个批次大小: {batch_size}")

# 查看一个批次的数据
train_iter = iter(train_loader)
batch_images, batch_labels = next(train_iter)
print(f"批次图像张量形状: {batch_images.shape}")
print(f"批次标签张量形状: {batch_labels.shape}")
print(batch_labels)

训练集DataLoader批次数: 35
测试集DataLoader批次数: 9
每个批次大小: 32
批次图像张量形状: torch.Size([32, 3, 128, 128])
批次标签张量形状: torch.Size([32])
tensor([6, 6, 3, 6, 7, 5, 1, 0, 4, 0, 5, 8, 9, 8, 6, 3, 4, 0, 8, 4, 8, 3, 2, 4,
        0, 5, 8, 3, 3, 0, 6, 3])


# 搭建模型

In [23]:
import torch.nn as nn

class MonkeyNet(nn.Module):
    def __init__(self, num_classes=10):
        super(MonkeyNet, self).__init__()

        # 针对10-monkeys高分辨率彩色图片(3通道，128x128)，构建更深更宽更正则化的卷积网络
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),   # (B,32,128,128)
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False),  # (B,32,128,128)
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                                       # (B,32,64,64)

            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),  # (B,64,64,64)
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),  # (B,64,64,64)
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                                       # (B,64,32,32)

            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False), # (B,128,32,32)
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1, bias=False),# (B,128,32,32)
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                                       # (B,128,16,16)

            nn.Conv2d(128, 256, kernel_size=3, padding=1, bias=False), # (B,256,16,16)
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1, bias=False),# (B,256,16,16)
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2,2),                                     # (B,256,8,8)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = self.classifier(x)
        return x

# 实例化模型
model = MonkeyNet(num_classes=10)


In [24]:
# 使用随机输入对模型进行一次前向计算以验证模型结构是否正确
import torch

dummy_input = torch.randn(32, 3, 128, 128) 
output = model(dummy_input) #前向传播/前向计算/正向传播
print(f"Output shape: {output.shape}")


Output shape: torch.Size([32, 10])


In [25]:
# 输出model每一层的参数量
total_params = 0  # 初始化总参数量为0
print("各层参数量统计：")  # 打印参数统计表头
for name, param in model.named_parameters():  # 遍历模型中所有需要优化的参数
    if param.requires_grad:  # 只有需要梯度更新的参数才统计
        num_params = param.numel()  # 计算当前参数的元素总数
        total_params += num_params  # 更新总参数量
        print(f"{name}: {num_params}")  # 输出当前层的参数量
print(f"模型总参数量: {total_params}")  # 输出模型总参数量


各层参数量统计：
features.0.weight: 864
features.1.weight: 32
features.1.bias: 32
features.3.weight: 9216
features.4.weight: 32
features.4.bias: 32
features.7.weight: 18432
features.8.weight: 64
features.8.bias: 64
features.10.weight: 36864
features.11.weight: 64
features.11.bias: 64
features.14.weight: 73728
features.15.weight: 128
features.15.bias: 128
features.17.weight: 147456
features.18.weight: 128
features.18.bias: 128
features.21.weight: 294912
features.22.weight: 256
features.22.bias: 256
features.24.weight: 589824
features.25.weight: 256
features.25.bias: 256
classifier.1.weight: 32768
classifier.1.bias: 128
classifier.4.weight: 1280
classifier.4.bias: 10
模型总参数量: 1207402


# 训练

In [26]:
import torch.nn as nn
import torch.optim as optim

# 初始化交叉熵损失函数，内部会做softmax
criterion = nn.CrossEntropyLoss()

# 初始化优化器（这里选用Adam，也可以使用SGD等）
optimizer = optim.Adam(model.parameters())


In [27]:
import trainmodule_train
# 假设train_loader和val_loader已定义，device已经设为"cuda"或"cpu"
trainer = trainmodule_train.Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    eval_step=100
)

# 设定训练轮数
num_epochs = 20

# 开始训练
trainer.train(num_epochs)


Epoch [1/20]  Train Loss: 2.0495  Train Acc: 0.2525
Epoch [2/20]  Train Loss: 1.8117  Train Acc: 0.3373
[Step 100] Val Loss: 1.7101 Val Acc: 0.3824
Epoch [3/20]  Train Loss: 1.7353  Train Acc: 0.3674
Epoch [4/20]  Train Loss: 1.6547  Train Acc: 0.3801
Epoch [5/20]  Train Loss: 1.5748  Train Acc: 0.4166
[Step 200] Val Loss: 1.4983 Val Acc: 0.4338
Epoch [6/20]  Train Loss: 1.6152  Train Acc: 0.3902
Epoch [7/20]  Train Loss: 1.5332  Train Acc: 0.4157
Epoch [8/20]  Train Loss: 1.5003  Train Acc: 0.4139
[Step 300] Val Loss: 1.3897 Val Acc: 0.5110
Epoch [9/20]  Train Loss: 1.4593  Train Acc: 0.4531
Epoch [10/20]  Train Loss: 1.4818  Train Acc: 0.4266
Epoch [11/20]  Train Loss: 1.3863  Train Acc: 0.4640
[Step 400] Val Loss: 1.3633 Val Acc: 0.5037
Epoch [12/20]  Train Loss: 1.3177  Train Acc: 0.4968
Epoch [13/20]  Train Loss: 1.3478  Train Acc: 0.4731
Epoch [14/20]  Train Loss: 1.2971  Train Acc: 0.5141
[Step 500] Val Loss: 1.3469 Val Acc: 0.5184
Epoch [15/20]  Train Loss: 1.3808  Train Acc: 0